# <span style="color:blue">Optuna UNet 2D</span>

**Content under Creative Commons Attribution license CC-BY-NC-SA 4.0**  
**Code under GNU-GPL v3 License**  
**© 2024 Francesco Chiumento**

This notebook facilitates the optimization of the 2D UNet neural network. It enables the optimization of both hyperparameters and data augmentation values. In the current setup, data augmentation parameters have already been optimized, and the focus is on optimizing hyperparameters.

Several loss functions from MONAI, including DiCELoss, DiceCELoss, GeneralizedDiceLoss, and FocalLoss, can be explored. However, BCEWithLogitsLoss has been found to yield optimal results for this neural network and is used as the loss function for hyperparameter optimization.

---

This notebook is part of the paper: 
> Chiumento F. et al. *Reducing Annotation Burden for Femoral Cartilage Segmentation in Knee MRI via Cross-Sequence Transfer Learning*  
> Preprint:  
> Submitted for peer review

---

- Imports:

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from model import UNET
from utils import get_loaders, check_accuracy, save_checkpoint
from train import train_fn
from monai.losses import DiceLoss, DiceCELoss, GeneralizedDiceLoss, FocalLoss
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import logging
import os
from concurrent.futures import ThreadPoolExecutor
import hashlib

- Variables:

In [ ]:
# Setup for Optuna's pruner
pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5, interval_steps=1)

logging.basicConfig(level=logging.INFO)

# Constants
CHECKPOINT_DIR = "checkpoints/"
SQLITE_DB_PATH = "sqlite:///example.db"
db_path = "example.db"

# Check if the database file exists
if os.path.exists(db_path):
    print(f"The file {db_path} exists.")
else:
    print(f"The file {db_path} does not exist.")

# Create checkpoint directory if it doesn't exist
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

# List available checkpoints
checkpoints = os.listdir(CHECKPOINT_DIR)
if checkpoints:
    print("Checkpoints found:")
    for checkpoint in checkpoints:
        print(checkpoint)
else:
    print("No checkpoints found.")

logging.info(f"Using SQLite database at {SQLITE_DB_PATH}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_IMG_DIR = "data/train_images"
TRAIN_MASK_DIR = "data/train_masks"
VAL_IMG_DIR = "data/val_images"
VAL_MASK_DIR = "data/val_masks"
NUM_WORKERS = 20
PIN_MEMORY = True

# Options for model features and loss functions
FEATURES_OPTIONS = ['64,128,256,512']
LOSS_FUNCTION_OPTIONS = ['BCEWithLogitsLoss']

- Functions:

In [ ]:
def to_float32(image, **kwargs):
    """
    Convert image to float32.

    Parameters
    ----------
    image : np.ndarray
        The input image.

    Returns
    -------
    np.ndarray
        The image converted to float32.
    """
    return image.astype(np.float32)

In [ ]:
# Training data augmentation and normalization
train_transform = A.Compose(
    [
        A.Resize(height=192, width=192),
        A.Rotate(limit=16, p=0.5),
        A.HorizontalFlip(p=0.5),
        A.ElasticTransform(alpha=23.59237990289956, sigma=5.615320305420207, p=0.5),
        A.RandomBrightnessContrast(p=0.12765906010940886),
        A.GaussianBlur(blur_limit=(7, 11), p=0.1),
        A.Lambda(image=to_float32, mask=to_float32),
        A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0, always_apply=True),
        ToTensorV2(),
    ]
)

# Validation data normalization
val_transform = A.Compose(
    [
        A.Resize(height=192, width=192),
        A.Lambda(image=to_float32, mask=to_float32),
        A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0], max_pixel_value=255.0, always_apply=True),
        ToTensorV2(),
    ]
)

In [ ]:
# Asynchronously save checkpoint
def save_checkpoint_async(state, filename):
    """
    Save checkpoint asynchronously.

    Parameters
    ----------
    state : dict
        State dictionary containing model state and optimizer state.
    filename : str
        The path where the checkpoint will be saved.

    Returns
    -------
    concurrent.futures.Future
        Future object representing the asynchronous save operation.
    """
    with ThreadPoolExecutor() as executor:
        future = executor.submit(torch.save, state, filename)
    try:
        future.result()
        logging.info(f"Checkpoint saved successfully to {filename}")
    except Exception as e:
        logging.error(f"Error saving checkpoint to {filename}: {e}")
    return future

In [ ]:
# Generate directory for a trial based on its parameters
def generate_trial_dir(trial):
    """
    Generate directory name for a trial based on its parameters.

    Parameters
    ----------
    trial : optuna.trial.Trial
        The trial object containing the parameters.

    Returns
    -------
    str
        The path of the generated trial directory.
    """
    lr = trial.params.get('lr')
    batch_size = trial.params.get('batch_size')
    features_option = trial.params.get('features')
    loss_function_name = trial.params.get('loss_function')

    config_str = f"{lr}_{batch_size}_{features_option}_{loss_function_name}"
    hash_object = hashlib.md5(config_str.encode())
    trial_dir = hash_object.hexdigest()
    print(f"Generated trial directory: {trial_dir}")
    return os.path.join(CHECKPOINT_DIR, trial_dir)

In [ ]:
# Load checkpoint from file
def load_checkpoint(checkpoint_path, model, optimizer):
    """
    Load checkpoint from file.

    Parameters
    ----------
    checkpoint_path : str
        The path to the checkpoint file.
    model : torch.nn.Module
        The model to load the state into.
    optimizer : torch.optim.Optimizer
        The optimizer to load the state into.

    Returns
    -------
    int
        The epoch number from which to resume.
    float
        The best dice score from the checkpoint.
    """
    print(f"Attempting to load checkpoint from {checkpoint_path}")
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path)
        model.load_state_dict(checkpoint['state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer'])
        logging.info(f"Checkpoint loaded from {checkpoint_path}")
        return checkpoint.get('epoch', 0), checkpoint.get('best_dice_score', 0.0)
    else:
        logging.warning(f"Checkpoint {checkpoint_path} not found")
        return 0, 0.0
 

In [ ]:
# Find the most recent trial
def find_last_trial(study):
    """
    Find the most recent trial in the study.

    Parameters
    ----------
    study : optuna.study.Study
        The Optuna study object.

    Returns
    -------
    optuna.trial.FrozenTrial or None
        The most recent trial or None if no trials exist.
    """
    if not study.trials:
        return None

    last_trial = max(study.trials, key=lambda t: t.number)
    return last_trial

In [ ]:
# Print the best trial details
def print_best_trial(study, trial):
    """
    Print the details of the best trial so far.

    Parameters
    ----------
    study : optuna.study.Study
        The Optuna study object.
    trial : optuna.trial.FrozenTrial
        The trial object to print details of.
    """
    if len(study.trials) > 0 and any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
        best_trial = study.best_trial
        print('Best trial so far:')
        print(f'  Value: {best_trial.value}')
        print('  Params: ')
        for key, value in best_trial.params.items():
            print(f'    {key}: {value}')
    else:
        print("No completed trials found.")

In [ ]:
#Objective function for Optuna optimization
def objective(trial, start_epoch=0, best_dice_score=0.0, checkpoint_path=None):
    """
    Objective function for Optuna optimization.

    Parameters
    ----------
    trial : optuna.trial.Trial
        The trial object.
    start_epoch : int, optional
        The starting epoch number, by default 0.
    best_dice_score : float, optional
        The best dice score so far, by default 0.0.
    checkpoint_path : str, optional
        The path to the checkpoint file, by default None.

    Returns
    -------
    float
        The best dice score achieved during the trial.
    """
    lr = trial.suggest_float('lr', 0.000019218433499427676, 0.0019218433499427676, log=True)
    batch_size = trial.suggest_int('batch_size', 1, 8)
    num_epochs = trial.suggest_int('num_epochs', 5, 35)
    features_option = trial.suggest_categorical('features', FEATURES_OPTIONS)
    features = tuple(map(int, features_option.split(',')))
    loss_function_name = trial.suggest_categorical('loss_function', LOSS_FUNCTION_OPTIONS)
    dropout_rate = trial.suggest_float('dropout_rate', 0.15, 0.25)

    print(f'Trial {trial.number}:')
    print(f'  Learning Rate: {lr}')
    print(f'  Batch Size: {batch_size}')
    print(f'  Number of Epochs: {num_epochs}')
    print(f'  Features: {features}')
    print(f'  Loss Function: {loss_function_name}')
    print(f'  Dropout Rate: {dropout_rate}')

    if loss_function_name == 'BCEWithLogitsLoss':
        loss_fn = nn.BCEWithLogitsLoss()
    elif loss_function_name == 'FocalLoss':
        loss_fn = FocalLoss(gamma=3.0)
    elif loss_function_name == 'DiceLoss':
        loss_fn = DiceLoss(sigmoid=True)
    else:
        logging.error(f"Loss function '{loss_function_name}' not recognized.")
        return 0.0

    model = UNET(in_channels=3, out_channels=1, features=features, dropout_rate=dropout_rate).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler = torch.cuda.amp.GradScaler()
    
    # Get data loader
    train_loader, val_loader = get_loaders(
        TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR,
        batch_size, train_transform, val_transform, NUM_WORKERS, PIN_MEMORY
    )

    # Load from checkpoint if available
    if checkpoint_path and os.path.exists(checkpoint_path):
        start_epoch, best_dice_score = load_checkpoint(checkpoint_path, model, optimizer)
        logging.info(f"Resuming training from epoch {start_epoch} with best dice score {best_dice_score}")
    else:
        logging.info(f"No valid checkpoint found at {checkpoint_path}, starting from scratch.")


    checkpoint_dir = generate_trial_dir(trial)
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    checkpoint_path = os.path.join(checkpoint_dir, "checkpoint.pth.tar")

    # Training loop
    for epoch in range(start_epoch, num_epochs):
        print(f"Starting epoch {epoch+1}/{num_epochs}")
        train_fn(train_loader, model, optimizer, loss_fn, scaler)

        print(f"Memory allocated: {torch.cuda.memory_allocated(DEVICE) / 1024 ** 3:.2f} GB")
        print(f"Max memory allocated: {torch.cuda.max_memory_allocated(DEVICE) / 1024 ** 3:.2f} GB")
        print(f"Memory cached: {torch.cuda.memory_reserved(DEVICE) / 1024 ** 3:.2f} GB")

        print("Starting Dice calculation")
        current_dice_score = check_accuracy(val_loader, model, device=DEVICE)
        print(f"Epoch {epoch+1}/{num_epochs} - Dice Score: {current_dice_score}")

        logging.info(f"Trial {trial.number} - Epoch {epoch+1}/{num_epochs} - Dice Score: {current_dice_score}")

        trial.report(current_dice_score, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        # Save checkpoint if current Dice score is the best
        if current_dice_score > best_dice_score:
            best_dice_score = current_dice_score
            checkpoint = {
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch + 1,
                "best_dice_score": best_dice_score
            }
            future = save_checkpoint_async(checkpoint, checkpoint_path)
            print(f"Checkpoint saved with better Dice: {current_dice_score}")

    print(f'Trial {trial.number} returning Dice Score: {best_dice_score}')

    return best_dice_score

In [ ]:
# Print details of all trial
def print_all_trials(study):
    """
    Print details of all trials in the study.

    Parameters
    ----------
    study : optuna.study.Study
        The Optuna study object.
    """
    trials = study.trials
    for trial in trials:
        print(f"Trial {trial.number}:")
        print(f"  State: {trial.state}")
        print(f"  Value: {trial.value}")
        print(f"  Params: {trial.params}")

- Main function:

In [ ]:
if __name__ == '__main__':
    logging.info("Starting Optuna study...")
    if os.path.exists("example.db"):
        logging.info("Found existing SQLite database, resuming study...")
    else:
        logging.info("No existing SQLite database found, starting new study...")

    study_name = "optuna_study"
    study = optuna.create_study(study_name=study_name, direction='maximize', storage=SQLITE_DB_PATH, load_if_exists=True, pruner=pruner)

    # Calculate remaining trials
    completed_or_pruned_trials = [t for t in study.trials if t.state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED]]
    num_completed_or_pruned_trials = len(completed_or_pruned_trials)
    total_trials = 50
    remaining_trials = total_trials - num_completed_or_pruned_trials

    logging.info(f"{num_completed_or_pruned_trials} trials already completed or pruned. {remaining_trials} trials remaining.")

    last_trial = find_last_trial(study)
    if last_trial:
        if last_trial.state == optuna.trial.TrialState.RUNNING:
            print(f"Resuming trial {last_trial.number}")
            try:
                checkpoint_dir = generate_trial_dir(last_trial)
                checkpoint_path = os.path.join(checkpoint_dir, "checkpoint.pth.tar")
                print(f"Checkpoint path: {checkpoint_path}")

                features_option = last_trial.params['features']
                features = tuple(map(int, features_option.split(',')))
                dropout_rate = last_trial.params['dropout_rate']
                model = UNET(in_channels=3, out_channels=1, features=features, dropout_rate=dropout_rate).to(DEVICE)
                optimizer = optim.Adam(model.parameters(), lr=last_trial.params['lr'])

                start_epoch, best_dice_score = load_checkpoint(checkpoint_path, model, optimizer)
                objective(last_trial, start_epoch=start_epoch, best_dice_score=best_dice_score, checkpoint_path=checkpoint_path)

                completed_or_pruned_trials = [t for t in study.trials if t.state in [optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED]]
                num_completed_or_pruned_trials = len(completed_or_pruned_trials)
                remaining_trials = total_trials - num_completed_or_pruned_trials
            except Exception as e:
                logging.error(f"Error in trial {last_trial.number}: {e}")

        else:
            logging.info(f"Last trial {last_trial.number} was completed or pruned. Starting new trials from {last_trial.number + 1}.")
            try:
                study.optimize(objective, n_trials=remaining_trials, timeout=None, callbacks=[print_best_trial])
                remaining_trials = 0  
            except Exception as e:
                logging.error(f"Optimization interrupted: {e}")

    # Start new trials if remaining
    if remaining_trials > 0:
        try:
            study.optimize(objective, n_trials=remaining_trials, timeout=None, callbacks=[print_best_trial])
        except Exception as e:
            logging.error(f"Optimization interrupted: {e}")
    elif not last_trial:
        logging.info("No trials found. Starting fresh study.")
        try:
            study.optimize(objective, n_trials=total_trials, timeout=None, callbacks=[print_best_trial])
        except Exception as e:
            logging.error(f"Optimization interrupted: {e}")
            
     # Print the best trial details if available
    if len(study.trials) > 0 and any(trial.state == optuna.trial.TrialState.COMPLETE for trial in study.trials):
        print('Best trial:')
        trial = study.best_trial

        print(f'  Value: {trial.value}')
        print('  Params: ')
        for key, value in trial.params.items():
            print(f'    {key}: {value}')
    else:
        print("No completed trials found.")

    print_all_trials(study)

---
## Dependencies

In [ ]:
%load_ext watermark

%watermark
%watermark --iversions

---
<a name="attribution"></a>

Notebook created using the [template](https://github.com/ORMIRcommunity/templates/blob/main/ORMIR_nb_template.ipynb) of the [ORMIR community](https://ormircommunity.github.io/) (version 1.0, 2023)